# Sample Model
**Regression vs Classification (XGBoost)**


**Dataset**
- A small synthetic dataset with three features:
  - `age`
  - `income_k` (income in thousands)
  - `tenure_years` (years with a company or service)
- A numeric target `target` generated from a simple linear formula plus noise.

**Regression Task**
- Goal: predict the continuous `target` value from the three features.
- Output: a single numeric prediction.
- Example targets: price, revenue, time-to-complete.
- Typical output: a single number, e.g. `64091.98`.

**Classification Task (XGBoost)**
- Goal: convert `target` into a binary label (e.g., above vs. below median) and predict that class.
- Output: a probability and a derived class label (0/1).
- Typical outputs:
  - **Probability** for class 1 (e.g., `0.0031`)
  - **Label** derived from the probability using a threshold (commonly `0.5`).

**Why there is a `label` for XGBoost**
- The model produces a probability.
- You convert it to a class using a rule like:
  - `label = 1 if probability >= 0.5 else 0`
- The label is the final decision your app can act on.

**Quick examples**
```text
Regression output:  {"prediction": 64091.98}

Classification output: {"probability": 0.0031, "label": 0}


In [31]:
import os
import tarfile
import joblib
import numpy as np
from sklearn.linear_model import LinearRegression
from xgboost import XGBClassifier
import json
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
from google.oauth2 import service_account

### Defining & Saving A Dummy Dataset

In [20]:
data_path = "../Data/dummy_data.csv"

# Generate dummy data
rng = np.random.default_rng(42)
n = 200
age = rng.integers(18, 70, size=n)
income_k = rng.normal(70, 15, size=n).clip(30, 120)  # in thousands
tenure_years = rng.integers(0, 10, size=n)

# Target variable (linear-ish with noise)
y = (
    10000
    + 120 * age
    + 500 * income_k
    + 800 * tenure_years
    + rng.normal(0, 3000, size=n)
)

dummy_data = pd.DataFrame({
    "age": age,
    "income_k": income_k,
    "tenure_years": tenure_years,
    "target": y,
})

dummy_data.to_csv(data_path, index=False)
data_path


'../Data/dummy_data.csv'

# Regression Model Training

### Training & Saving Model

In [24]:
data_reg = "../Data/dummy_data.csv"
df_regression = pd.read_csv(data_reg)

os.makedirs("../Inference/Models/regression", exist_ok=True)

# Features + target
X_reg = df_regression[["age", "income_k", "tenure_years"]]
y_reg = df_regression["target"]

# Train
model_reg = LinearRegression()
model_reg.fit(X_reg, y_reg)

# Save model
model_path_reg = "./regression_model.joblib"
joblib.dump(model_reg, model_path_reg)

model_path_reg

'../Inference/Models/regression/regression_model.joblib'

# Testing Models

In [36]:
def _coerce_payload(payload):
    if isinstance(payload, str):
        return json.loads(payload)
    if isinstance(payload, dict):
        return payload
    raise ValueError("payload must be a dict or JSON string")

def _as_float(payload, key):
    if key not in payload:
        raise ValueError(f"missing field: {key}")
    try:
        return float(payload[key])
    except Exception as exc:
        raise ValueError(f"{key} must be numeric") from exc
    
def predict_regression(payload, model):
    p = _coerce_payload(payload)
    age = _as_float(p, "age")
    income_k = _as_float(p, "income_k")
    tenure_years = _as_float(p, "tenure_years")

    X = pd.DataFrame([{
        "age": age,
        "income_k": income_k,
        "tenure_years": tenure_years
    }])

    pred = float(model.predict(X)[0])
    return {"prediction": pred}

In [37]:
# Load models only if not already defined
try:
    model_reg
except NameError:
    model_reg = joblib.load("./regression_model.joblib")

In [38]:
# Sample payloads
payload_reg = {"age": 42, "income_k": 88.0, "tenure_years": 6}

# Call your functions
print(predict_regression(payload_reg, model_reg))

# Also test with JSON strings
print(predict_regression(json.dumps(payload_reg), model_reg))


{'prediction': 64091.976420931634}
{'probability': 0.003137440187856555, 'label': 0}
{'prediction': 64091.976420931634}
{'probability': 0.003137440187856555, 'label': 0}
